In [0]:
from pyspark.sql.functions import lit
from pyspark.sql import DataFrame
from functools import reduce
from datetime import date

catalog = 'operations'
schema = 'finance_staging'

volume_base = "/Volumes/operations/finance_staging/edgar_data"

current_year = date.today().year
years = [str(y) for y in range(2019, current_year + 1)]
quarters = ['q1', 'q2', 'q3', 'q4']
files = ['pre', 'num', 'sub', 'tag']

def path_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

for file in files:
    partition_dfs = []

    for year in years:
        for quarter in quarters:
            filing_path = f"{volume_base}/{year}/{quarter}/{file}.txt"

            if not path_exists(filing_path):
                print(f"Skipping {year} {quarter.upper()} {file} — file not found.")
                continue

            filing = (
                spark.read
                .option("sep", "\t")
                .option("header", "true")
                .option("inferSchema", "true")
                .option("nullValue", "")
                .csv(filing_path)
                .withColumn("source_file", lit(filing_path))
                .withColumn("source_file_description", lit(f"{year}_{quarter}_{file}"))
            )
            partition_dfs.append(filing)

    if not partition_dfs:
        print(f"No data found for {file}, skipping table write.")
        continue

    combined = reduce(DataFrame.union, partition_dfs)

    (
        combined.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f'{catalog}.{schema}.raw_{file}_tbl')
    )

    print(f"Written: {catalog}.{schema}.raw_{file}_tbl")